# 03 — OCR toàn bộ keyframe (resume theo shard)

**Phụ trách:** Tuấn Anh (N1), Quang Dũng hỗ trợ chia ca · **Mốc:** M2 · **GPU:** nên có (T4)

Chữ trên màn hình (headline tin tức, biển hiệu, phụ đề) là kênh tìm kiếm giá trị cao.
Kết quả: file `ocr_shard_*.jsonl` trên Drive → N1 gộp vào BM25.

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
# PaddleOCR trên Colab: nếu bản GPU lỗi thì đổi sang 'paddlepaddle' (CPU, chậm hơn ~4x)
%pip -q install paddlepaddle-gpu paddleocr

# === CONFIG ===
DRIVE_ROOT   = '/content/drive/MyDrive/AIC2026'
KEYFRAME_DIR = f'{DRIVE_ROOT}/data/keyframes'
OUT_DIR      = f'{DRIVE_ROOT}/outputs/ocr_shards'
SHARD_SIZE   = 1000
SHARD_START, SHARD_END = 0, 9999   # ← MỖI TÀI KHOẢN SỬA DẢI NÀY
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# === OCR theo shard ===
from paddleocr import PaddleOCR
import glob, os, json

ocr = PaddleOCR(use_angle_cls=True, lang='vi', show_log=False)
images = sorted(glob.glob(f'{KEYFRAME_DIR}/*/*.jpg'))
n_shards = (len(images) - 1) // SHARD_SIZE + 1
print(f'{len(images)} ảnh, {n_shards} shard')

def kf_id(path):
    return f'{os.path.basename(os.path.dirname(path))}#{os.path.splitext(os.path.basename(path))[0]}'

for s in range(max(0, SHARD_START), min(n_shards, SHARD_END + 1)):
    out_f = f'{OUT_DIR}/ocr_shard_{s:05d}.jsonl'
    if os.path.exists(out_f):
        continue
    rows = []
    for p in images[s*SHARD_SIZE : (s+1)*SHARD_SIZE]:
        try:
            res = ocr.ocr(p, cls=True)
            texts = [line[1][0] for block in (res or []) if block for line in block
                     if line[1][1] > 0.6]           # bỏ text confidence thấp
            if texts:
                rows.append({'keyframe_id': kf_id(p), 'text': ' '.join(texts)})
        except Exception as e:
            rows.append({'keyframe_id': kf_id(p), 'error': str(e)})
    with open(out_f, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'✓ shard {s}/{n_shards} — {len(rows)} dòng có chữ')
print('XONG dải được giao.')